# 04 — Data Release Exploration

> **Audience:** Anyone who has downloaded the PBI Zenodo data release and wants
> to understand its contents.  No PBI Python package required — only `duckdb` and
> `pandas` (plus optional `matplotlib`/`seaborn` for plots).

| | |
|---|---|
| **Expected inputs** | Extracted Zenodo release archive (contains `duckdb/`, `hosts/`, `phages/`, `reports/`, `manifest.csv`) |
| **Outputs** | Summary tables and figures in `<results>/04_data_release_exploration/` |
| **Requirements** | `pip install duckdb pandas matplotlib seaborn` |
| **Companion** | `06_reproducibility.ipynb` for provenance and citation guidance |

## What this notebook covers

1. Verify the archive is intact (manifest + database schema)
2. Explore phage metadata: source distribution, genome length, GC content, lifestyle, completeness
3. Inspect annotation coverage per phage (proteins, terminators, AMR genes, CRISPR, …)
4. Browse host metadata and phage–host links
5. Navigate the data-merging HTML reports
6. Example cross-table SQL queries

## Data release structure

```text
.
+-- duckdb/
|   \-- phage_database_optimized.duckdb   <- star-schema metadata database
+-- hosts/                                 <- host genome metadata (CSV)
+-- phages/                                <- phage annotation metadata (CSV)
+-- reports/                               <- HTML data-merging / validation reports
\-- manifest.csv                           <- file inventory with checksums
```

> **Sequences not included.** Large FASTA files are omitted to keep the download
> manageable.  To retrieve nucleotide / protein sequences use the full PBI pipeline
> at <https://github.com/ThibaultSchowing/PBI>.


## Setup & Path Configuration

Set `DATA_RELEASE_ROOT` below to the directory that *contains* `duckdb/`, `hosts/`,
`phages/`, and `reports/`.  When running from *inside* the release archive the
default (`Path(".")`) is correct.

> **Provenance tip:** Check the `release_notes.txt` or `pipeline_run_provenance.json`
> (if included in the release) for the exact snapshot date and schema profile used to
> build this database.  See `06_reproducibility.ipynb` for a detailed walkthrough.


In [1]:
import json
import os
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── USER CONFIGURATION ──────────────────────────────────────────────────────
# Set this to the root of the data release (the folder that contains duckdb/,
# hosts/, phages/, reports/, and manifest.csv).
DATA_RELEASE_ROOT = Path(".")
# ────────────────────────────────────────────────────────────────────────────

DB_PATH      = DATA_RELEASE_ROOT / "duckdb" / "phage_database_optimized.duckdb"
PHAGES_DIR   = DATA_RELEASE_ROOT / "phages"
HOSTS_DIR    = DATA_RELEASE_ROOT / "hosts"
REPORTS_DIR  = DATA_RELEASE_ROOT / "reports"
MANIFEST_CSV = DATA_RELEASE_ROOT / "manifest.csv"

# Visualisation defaults
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 10
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 20)

print("Data release root:", DATA_RELEASE_ROOT.resolve())
print("DuckDB path      :", DB_PATH)
print("DB file exists   :", DB_PATH.exists())

results_root = Path(os.getenv('PBI_RESULTS_DIR', '/results'))
try:
    results_root.mkdir(parents=True, exist_ok=True)
except Exception:
    results_root = Path.cwd() / 'outputs'
    results_root.mkdir(parents=True, exist_ok=True)

NOTEBOOK_RESULTS_DIR = results_root / '04_data_release_exploration'
TABLES_DIR = NOTEBOOK_RESULTS_DIR / 'tables'
FIGURES_DIR = NOTEBOOK_RESULTS_DIR / 'figures'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Notebook results directory: {NOTEBOOK_RESULTS_DIR}")


Data release root: /workspace
DuckDB path      : duckdb/phage_database_optimized.duckdb
DB file exists   : False
Notebook results directory: /results/04_data_release_exploration


## 1. Manifest — File Inventory

The `manifest.csv` lists every file included in the release together with its size and checksum.  
It is useful for verifying download integrity and understanding what is available.

In [2]:
if MANIFEST_CSV.exists():
    manifest = pd.read_csv(MANIFEST_CSV)
    print(f"Manifest: {len(manifest)} files")
    display(manifest)
else:
    print("manifest.csv not found — skipping.")

manifest.csv not found — skipping.


## 2. DuckDB Database — Connection & Schema

The DuckDB file at `duckdb/phage_database_optimized.duckdb` is a **star-schema** relational database:

| Table | Description |
|---|---|
| `fact_phages` | Central fact table — one row per phage genome |
| `dim_proteins` | Annotated protein records per phage |
| `dim_terminators` | Predicted transcription terminators |
| `dim_anti_crispr` | Anti-CRISPR protein predictions |
| `dim_virulent_factors` | Virulence factor hits (VFDB) |
| `dim_transmembrane_proteins` | Transmembrane helix predictions (TMHMM) |
| `dim_trna_tmrna` | tRNA / tmRNA predictions |
| `dim_antimicrobial_resistance_genes` | AMR gene hits (RGI / CARD) |
| `dim_crispr_arrays` | CRISPR array predictions |
| `dim_hosts` | Host bacterial species metadata |
| `dim_assembly_metadata` | NCBI assembly-level metadata for host genomes |
| `dim_phage_host_links` | Phage–host infection links |

We open the database in **read-only** mode to avoid write locks.

In [3]:
# Open in read-only mode
conn = duckdb.connect(str(DB_PATH), read_only=True)
print("Connected to DuckDB:", DB_PATH.name)

# List all tables
tables = conn.execute("SHOW TABLES").fetchdf()
print(f"\nTables in database ({len(tables)}):")
for tbl in tables["name"]:
    print(" -", tbl)

IOException: IO Error: Cannot open database "/workspace/duckdb/phage_database_optimized.duckdb" in read-only mode: database does not exist

### 2.1 Row Counts — Database Overview

A quick row-count query across all tables gives a high-level picture of the dataset size.

> **Performance note:** Row counts are retrieved from DuckDB's internal catalog statistics (`duckdb_tables()`) to avoid full table scans on large tables (e.g. `dim_proteins` with tens of millions of rows).

In [ ]:
table_names = tables["name"].tolist()

# Use DuckDB's internal catalog for fast row-count estimates.
# This avoids full table scans on large tables (e.g. dim_proteins ~43 M rows).
try:
    print("\nQuerying DuckDB catalog for row count estimates...")
    catalog_df = conn.execute(
        "SELECT table_name, estimated_size FROM duckdb_tables()"
    ).fetchdf()
    row_counts = dict(zip(catalog_df["table_name"], catalog_df["estimated_size"]))
    # Fall back to exact count for any table not covered by the catalog
    #for tbl in table_names:
    #    if tbl not in row_counts:
    #        row_counts[tbl] = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
except Exception:
    # Fallback: exact counts (slow for very large tables)
    print("Catalog query failed — falling back to exact counts (may be slow):")
    row_counts = {}
    for tbl in table_names:
        row_counts[tbl] = conn.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]

counts_df = pd.DataFrame(
    [(tbl, cnt) for tbl, cnt in row_counts.items()],
    columns=["Table", "Row Count"]
).sort_values("Row Count", ascending=False)

print("Database row counts (estimated from catalog):")
print(counts_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(counts_df["Table"], counts_df["Row Count"], color=sns.color_palette("Blues_r", len(counts_df)))
ax.set_xlabel("Row Count")
ax.set_title("DuckDB Table Sizes (estimated)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
for bar, val in zip(bars, counts_df["Row Count"]):
    ax.text(bar.get_width() + max(counts_df["Row Count"]) * 0.005,
            bar.get_y() + bar.get_height() / 2,
            f"{val:,}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

## 3. Phage Metadata Exploration

The `fact_phages` table is the central table of the database.  
Each row represents one phage genome, annotated with its source database, taxonomy, host, and quality metrics.

In [ ]:
# Preview the schema and a few rows
print("Schema of fact_phages:")
display(conn.execute("DESCRIBE fact_phages").fetchdf())

print("\nSample rows:")
display(conn.execute("SELECT * FROM fact_phages LIMIT 5").fetchdf())

### 3.1 Phage Entries per Source Database

Phages are aggregated from several public databases.  
Understanding the source distribution reveals potential biases and coverage.

In [ ]:
source_df = conn.execute("""
    SELECT
        Source_DB,
        COUNT(*) AS phage_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM fact_phages
    GROUP BY Source_DB
    ORDER BY phage_count DESC
""").fetchdf()

print("Phage entries per source database:")
display(source_df)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
axes[0].bar(source_df["Source_DB"], source_df["phage_count"],
            color=sns.color_palette("Set2", len(source_df)))
axes[0].set_xlabel("Source Database")
axes[0].set_ylabel("Phage Count")
axes[0].set_title("Phages by Source Database")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha="right")

# Pie chart
axes[1].pie(source_df["phage_count"], labels=source_df["Source_DB"],
            autopct="%1.1f%%", colors=sns.color_palette("Set2", len(source_df)))
axes[1].set_title("Proportion by Source Database")

plt.suptitle("Source Database Distribution", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### 3.2 Genome Length and GC Content

Genome length (bp) and GC content (%) are fundamental quality metrics for phage genomes.

> **Performance note:** Histograms are drawn from a random reservoir sample of up to 50 000 rows so that plotting remains fast even for large tables.

In [ ]:
genome_stats = conn.execute("""
    SELECT
        COUNT(*)                          AS total_phages,
        COUNT(Length)                     AS with_length,
        MIN(Length)                       AS min_length,
        MAX(Length)                       AS max_length,
        ROUND(AVG(Length), 0)             AS avg_length,
        ROUND(MEDIAN(Length), 0)          AS median_length,
        COUNT(GC_content)                 AS with_gc,
        ROUND(MIN(GC_content), 2)         AS min_gc,
        ROUND(MAX(GC_content), 2)         AS max_gc,
        ROUND(AVG(GC_content), 2)         AS avg_gc
    FROM fact_phages
""").fetchdf()

print("Genome length and GC content summary:")
display(genome_stats.T.rename(columns={0: "Value"}))

# Sample up to 50 000 rows for histograms (avoids fetching millions of rows)
length_gc = conn.execute("""
    SELECT Length, GC_content
    FROM fact_phages
    WHERE Length IS NOT NULL AND GC_content IS NOT NULL
    USING SAMPLE reservoir(50000 ROWS)
""").fetchdf()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(length_gc["Length"], bins=100, color="steelblue", edgecolor="none")
axes[0].set_xlabel("Genome Length (bp)")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Phage Genome Length")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

axes[1].hist(length_gc["GC_content"], bins=60, color="seagreen", edgecolor="none")
axes[1].set_xlabel("GC Content (%)")
axes[1].set_ylabel("Count")
axes[1].set_title("Distribution of GC Content")

plt.suptitle("Phage Genome Metrics", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### 3.3 Lifestyle Distribution

Phages are classified as **lytic** (virulent — lyse the host) or **temperate** (lysogenic — integrate into the host genome).  
A large fraction may be unlabelled (`NULL` or unknown).

In [ ]:
lifestyle_df = conn.execute("""
    SELECT
        COALESCE(Lifestyle, 'Unknown') AS Lifestyle,
        COUNT(*)                        AS count
    FROM fact_phages
    GROUP BY Lifestyle
    ORDER BY count DESC
""").fetchdf()

display(lifestyle_df)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(lifestyle_df["Lifestyle"], lifestyle_df["count"],
       color=sns.color_palette("pastel", len(lifestyle_df)))
ax.set_xlabel("Lifestyle")
ax.set_ylabel("Count")
ax.set_title("Phage Lifestyle Distribution")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

### 3.4 Genome Completeness

Completeness is predicted by CheckV (for metagenomic contigs) or taken from source database annotations.

In [ ]:
completeness_df = conn.execute("""
    SELECT
        COALESCE(Completeness, 'Unknown') AS Completeness,
        COUNT(*) AS count
    FROM fact_phages
    GROUP BY Completeness
    ORDER BY count DESC
""").fetchdf()

display(completeness_df)

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(completeness_df["Completeness"], completeness_df["count"],
        color=sns.color_palette("muted", len(completeness_df)))
ax.set_xlabel("Count")
ax.set_title("Phage Genome Completeness")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

### 3.5 Top Phage Hosts

The `Host` column records the bacterial genus/species reported as host in the source database.

In [ ]:
top_hosts = conn.execute("""
    SELECT
        Host,
        COUNT(*) AS phage_count
    FROM fact_phages
    WHERE Host IS NOT NULL AND Host != '-'
    GROUP BY Host
    ORDER BY phage_count DESC
    LIMIT 20
""").fetchdf()

display(top_hosts)

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(top_hosts["Host"][::-1], top_hosts["phage_count"][::-1],
        color=sns.color_palette("Blues", len(top_hosts))[::-1])
ax.set_xlabel("Number of Phages")
ax.set_title("Top 20 Hosts by Number of Phages")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

### 3.6 Phage–Host Links

The `dim_phage_host_links` table records verified phage–host infection pairs used for downstream machine-learning tasks.

In [ ]:
if "dim_phage_host_links" in table_names:
    print("Schema of dim_phage_host_links:")
    display(conn.execute("DESCRIBE dim_phage_host_links").fetchdf())

    link_stats = conn.execute("""
        SELECT
            COUNT(*)                    AS total_links,
            COUNT(DISTINCT Phage_ID)    AS unique_phages,
            COUNT(DISTINCT Assembly_Accession) AS unique_host_assemblies
        FROM dim_phage_host_links
    """).fetchdf()
    print("\nPhage–host link statistics:")
    display(link_stats)

    print("\nSample rows:")
    display(conn.execute("SELECT * FROM dim_phage_host_links LIMIT 10").fetchdf())
else:
    print("dim_phage_host_links table not found in this database.")

## 4. Phage Annotation Files (`phages/` directory)

The `phages/` directory contains flat CSV files — one per annotation type — produced by merging data from all source databases.  
These files are also loaded into the DuckDB dimension tables above.

| File | DuckDB table | Description |
|---|---|---|
| `merged_phage_metadata.csv` | `fact_phages` | Core phage genome metadata |
| `merged_annotated_proteins_metadata.csv` | `dim_proteins` | Protein annotations (function, physicochemical properties) |
| `merged_transcription_terminator_metadata.csv` | `dim_terminators` | Rho-independent terminators (TransTermHP) |
| `merged_phage_anti_crispr_metadata.csv` | `dim_anti_crispr` | Anti-CRISPR protein predictions |
| `merged_phage_virulent_factor_metadata.csv` | `dim_virulent_factors` | Virulence factor hits (VFDB) |
| `merged_phage_transmembrane_protein_metadata.csv` | `dim_transmembrane_proteins` | Transmembrane topology (TMHMM) |
| `merged_phage_trna_tmrna_metadata.csv` | `dim_trna_tmrna` | tRNA / tmRNA predictions (ARAGORN) |
| `merged_antimicrobial_resistance_gene_metadata.csv` | `dim_antimicrobial_resistance_genes` | AMR gene hits (RGI / CARD) |
| `merged_crispr_array_metadata.csv` | `dim_crispr_arrays` | CRISPR array predictions (CRISPRCasFinder) |

In [ ]:
phage_csvs = sorted(PHAGES_DIR.glob("*.csv"))
print(f"Files in phages/ ({len(phage_csvs)}):")

phage_file_stats = []
for csv_path in phage_csvs:
    try:
        df = pd.read_csv(csv_path, nrows=0)  # load only header
        # Count rows via DuckDB (faster than Python line iteration for large files)
        try:
            row_count = conn.execute(
                f"SELECT COUNT(*) FROM read_csv_auto('{csv_path}', header=true)"
            ).fetchone()[0]
        except Exception:
            with open(csv_path, "rb") as f:
                row_count = sum(1 for _ in f) - 1  # subtract header
        phage_file_stats.append({
            "File": csv_path.name,
            "Rows": row_count,
            "Columns": len(df.columns),
            "Size (MB)": round(csv_path.stat().st_size / 1024**2, 2)
        })
    except Exception as e:
        phage_file_stats.append({"File": csv_path.name, "Rows": "error",
                                  "Columns": "error", "Size (MB)": "error"})

phage_files_df = pd.DataFrame(phage_file_stats)
display(phage_files_df)

### 4.1 Preview: Core Phage Metadata CSV

In [ ]:
phage_meta_csv = PHAGES_DIR / "merged_phage_metadata.csv"
if phage_meta_csv.exists():
    phage_meta = pd.read_csv(phage_meta_csv, nrows=5)
    print(f"Columns ({len(phage_meta.columns)}): {list(phage_meta.columns)}")
    display(phage_meta)
else:
    print(f"{phage_meta_csv} not found.")

### 4.2 Preview: Annotated Proteins

In [ ]:
proteins_csv = PHAGES_DIR / "merged_annotated_proteins_metadata.csv"
if proteins_csv.exists():
    proteins_df = pd.read_csv(proteins_csv, nrows=5)
    print(f"Columns ({len(proteins_df.columns)}): {list(proteins_df.columns)}")
    display(proteins_df)
else:
    print(f"{proteins_csv} not found.")

### 4.3 Annotation Coverage per Phage

Not every phage has entries in every annotation table.  
This query shows what fraction of phages have at least one record in each dimension table.

In [ ]:
total_phages = conn.execute("SELECT COUNT(*) FROM fact_phages").fetchone()[0]

annotation_tables = [
    ("dim_proteins",                        "Annotated proteins"),
    ("dim_terminators",                     "Transcription terminators"),
    ("dim_anti_crispr",                     "Anti-CRISPR proteins"),
    ("dim_virulent_factors",                "Virulence factors"),
    ("dim_transmembrane_proteins",          "Transmembrane proteins"),
    ("dim_trna_tmrna",                      "tRNA / tmRNA"),
    ("dim_antimicrobial_resistance_genes",  "AMR genes"),
    ("dim_crispr_arrays",                   "CRISPR arrays"),
]

coverage_rows = []
for tbl, label in annotation_tables:
    if tbl in table_names:
        # Use an estimate for very large tables to avoid slow COUNT(DISTINCT) scans
        tbl_rows = row_counts.get(tbl, 0)
        if tbl_rows > 5_000_000:
            # Approximate via 10 % sample (fast, representative)
            phages_annotated = conn.execute(
                f"SELECT COUNT(DISTINCT Phage_ID) FROM {tbl} USING SAMPLE 10 PERCENT"
            ).fetchone()[0]
            phages_annotated = int(phages_annotated * 10)  # scale back up
        else:
            phages_annotated = conn.execute(
                f"SELECT COUNT(DISTINCT Phage_ID) FROM {tbl}"
            ).fetchone()[0]
        coverage_rows.append({
            "Annotation": label,
            "Phages with annotation": phages_annotated,
            "Coverage (%)": round(phages_annotated / total_phages * 100, 1)
        })

coverage_df = pd.DataFrame(coverage_rows).sort_values("Coverage (%)", ascending=False)
display(coverage_df)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(coverage_df["Annotation"][::-1], coverage_df["Coverage (%)"][::-1],
        color=sns.color_palette("Greens", len(coverage_df)))
ax.set_xlabel("Coverage (%)")
ax.set_title("Fraction of Phages with Each Annotation Type")
ax.set_xlim(0, 110)
for i, (val, label) in enumerate(zip(coverage_df["Coverage (%)"][::-1], coverage_df["Annotation"][::-1])):
    ax.text(val + 1, i, f"{val}%", va="center", fontsize=9)
plt.tight_layout()
plt.show()

## 5. Host Files (`hosts/` directory)

The `hosts/` directory contains metadata about the **bacterial host genomes** used for phage–host interaction analysis.

| File | Description |
|---|---|
| `host_metadata.csv` | Species-level host metadata (taxonomy, source DB) |
| `assembly_metadata.csv` | NCBI assembly metadata for downloaded host genomes |
| `phage_host_links.csv` | Phage–host infection links (IDs only) |
| `phage_host_assemblies.csv` | Phage–host pairs with assembly-level detail |
| `phage_host_candidates.csv` | Candidate host–phage pairs before filtering |
| `host_download_status.json` | Download status for each host genome |
| `host_fasta_mapping.json` | Mapping of host IDs to local FASTA file paths |

In [ ]:
host_csvs = sorted(HOSTS_DIR.glob("*.csv"))
host_jsons = sorted(HOSTS_DIR.glob("*.json"))

print(f"CSV files in hosts/ ({len(host_csvs)}):")
host_file_stats = []
for csv_path in host_csvs:
    try:
        df = pd.read_csv(csv_path, nrows=0)
        # Count rows via DuckDB (faster than Python line iteration for large files)
        try:
            row_count = conn.execute(
                f"SELECT COUNT(*) FROM read_csv_auto('{csv_path}', header=true)"
            ).fetchone()[0]
        except Exception:
            with open(csv_path, "rb") as f:
                row_count = sum(1 for _ in f) - 1
        host_file_stats.append({
            "File": csv_path.name,
            "Rows": row_count,
            "Columns": len(df.columns),
            "Size (MB)": round(csv_path.stat().st_size / 1024**2, 2)
        })
    except Exception as e:
        host_file_stats.append({"File": csv_path.name, "Rows": "error",
                                 "Columns": "error", "Size (MB)": "error"})

display(pd.DataFrame(host_file_stats))

print(f"\nJSON files in hosts/ ({len(host_jsons)}):")
for j in host_jsons:
    size_kb = j.stat().st_size / 1024
    print(f"  {j.name:40s}  {size_kb:.1f} KB")

### 5.1 Host Metadata

In [ ]:
host_meta_csv = HOSTS_DIR / "host_metadata.csv"
if host_meta_csv.exists():
    host_meta = pd.read_csv(host_meta_csv, nrows=1000)
    print(f"Host metadata: {len(host_meta):,} rows loaded (limited to 1 000 for preview), ")
    print(f"Columns ({len(host_meta.columns)}): {list(host_meta.columns)}")
    display(host_meta.head())
else:
    print(f"{host_meta_csv} not found.")

### 5.2 Assembly Metadata

In [ ]:
assembly_csv = HOSTS_DIR / "assembly_metadata.csv"
if assembly_csv.exists():
    assembly_meta = pd.read_csv(assembly_csv, nrows=1000)
    print(f"Assembly metadata: {len(assembly_meta):,} rows loaded (limited to 1 000 for preview)")
    print(f"Columns ({len(assembly_meta.columns)}): {list(assembly_meta.columns)}")
    display(assembly_meta.head())
else:
    print(f"{assembly_csv} not found.")

### 5.3 Phage–Host Links

In [ ]:
links_csv = HOSTS_DIR / "phage_host_links.csv"
if links_csv.exists():
    links_df = pd.read_csv(links_csv, nrows=1000)
    print(f"Phage–host links: {len(links_df):,} rows loaded (limited to 1 000 for preview)")
    print(f"Columns: {list(links_df.columns)}")
    display(links_df.head(10))
else:
    print(f"{links_csv} not found.")

### 5.4 Host Download Status (JSON)

The `host_download_status.json` records whether each host genome was successfully downloaded.

In [ ]:
status_json = HOSTS_DIR / "host_download_status.json"
if status_json.exists():
    with open(status_json) as f:
        download_status = json.load(f)

    statuses = list(download_status.values())
    from collections import Counter
    status_counts = Counter(str(v) for v in statuses)
    print(f"Total host genomes tracked: {len(download_status):,}")
    print("Status breakdown:")
    for status, cnt in sorted(status_counts.items(), key=lambda x: -x[1]):
        print(f"  {status}: {cnt:,}")
else:
    print(f"{status_json} not found.")

## 6. Data-Merging Reports (`reports/` directory)

The `reports/` directory contains **HTML reports** generated by the PBI pipeline.  
Each report corresponds to one annotation type and documents:
- The merging process across source databases
- Data quality checks (missing values, duplicates, outliers)
- Statistical summaries and column-level profiles

Open the HTML files in a web browser for an interactive view.

| Report file | Covers |
|---|---|
| `phage_metadata_report.html` | Core phage metadata (`merged_phage_metadata.csv`) |
| `annotated_proteins_metadata_report.html` | Protein annotations |
| `transcription_terminator_metadata_report.html` | Transcription terminators |
| `phage_anti_crispr_metadata_report.html` | Anti-CRISPR proteins |
| `phage_virulent_factor_metadata_report.html` | Virulence factors |
| `phage_transmembrane_protein_metadata_report.html` | Transmembrane topology |
| `phage_trna_tmrna_metadata_report.html` | tRNA / tmRNA |
| `antimicrobial_resistance_gene_metadata_report.html` | AMR genes |
| `crispr_array_metadata_report.html` | CRISPR arrays |
| `database_validation.html` | Full database integrity validation |

In [ ]:
report_files = sorted(REPORTS_DIR.glob("*.html"))
print(f"Reports in reports/ ({len(report_files)}):")
for rpt in report_files:
    size_kb = rpt.stat().st_size / 1024
    print(f"  {rpt.name:55s}  {size_kb:7.0f} KB")

In [ ]:
# Generate clickable links to open reports from this notebook
from IPython.display import display, HTML

report_links = []
for rpt in report_files:
    rel_path = os.path.relpath(rpt, Path("."))
    report_links.append(f'<li><a href="{rel_path}" target="_blank">{rpt.name}</a></li>')

display(HTML(
    "<b>Data-merging and validation reports (click to open):</b>"
    "<ul>" + "".join(report_links) + "</ul>"
))

> 🟡 **Optional deeper dive** — The queries below illustrate cross-table joins.
> Skip to the Summary if you only need the quick overview.

## 7. Example Queries

A few representative SQL queries to illustrate how to work with the database programmatically.

### 7.1 Phages with AMR Genes

In [ ]:
if "dim_antimicrobial_resistance_genes" in table_names:
    amr_phages = conn.execute("""
        SELECT
            p.Phage_ID,
            p.Source_DB,
            p.Host,
            p.Lifestyle,
            COUNT(a.Protein_ID) AS amr_gene_count
        FROM fact_phages p
        JOIN dim_antimicrobial_resistance_genes a USING (Phage_ID)
        GROUP BY p.Phage_ID, p.Source_DB, p.Host, p.Lifestyle
        ORDER BY amr_gene_count DESC
        LIMIT 15
    """).fetchdf()
    print(f"Top phages by number of AMR genes (showing top {len(amr_phages)}):")
    display(amr_phages)
else:
    print("dim_antimicrobial_resistance_genes table not found.")

### 7.2 Lytic Phages Infecting *Escherichia coli* with a Known Assembly

In [ ]:
if "dim_phage_host_links" in table_names and "dim_assembly_metadata" in table_names:
    ecoli_lytic = conn.execute("""
        SELECT
            p.Phage_ID,
            p.Length,
            p.GC_content,
            p.Lifestyle,
            l.Assembly_Accession
        FROM fact_phages p
        JOIN dim_phage_host_links l USING (Phage_ID)
        WHERE p.Host ILIKE '%Escherichia coli%'
          AND p.Lifestyle = 'lytic'
        LIMIT 10
    """).fetchdf()
    print(f"Lytic E. coli phages with known host assemblies ({len(ecoli_lytic)} shown):")
    display(ecoli_lytic)
else:
    # Fallback: query without assembly join
    ecoli_lytic = conn.execute("""
        SELECT Phage_ID, Length, GC_content, Lifestyle
        FROM fact_phages
        WHERE Host ILIKE '%Escherichia coli%'
          AND Lifestyle = 'lytic'
        LIMIT 10
    """).fetchdf()
    print(f"Lytic E. coli phages ({len(ecoli_lytic)} shown):")
    display(ecoli_lytic)

### 7.3 Protein Function Classification Distribution

In [ ]:
if "dim_proteins" in table_names:
    protein_classes = conn.execute("""
        SELECT
            COALESCE(Protein_classification, 'Unknown') AS classification,
            COUNT(*) AS protein_count
        FROM dim_proteins
        GROUP BY Protein_classification
        ORDER BY protein_count DESC
        LIMIT 15
    """).fetchdf()
    print("Top protein classifications:")
    display(protein_classes)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.barh(protein_classes["classification"][::-1],
            protein_classes["protein_count"][::-1],
            color=sns.color_palette("Oranges", len(protein_classes)))
    ax.set_xlabel("Protein Count")
    ax.set_title("Top Protein Function Classifications")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    plt.tight_layout()
    plt.show()
else:
    print("dim_proteins table not found.")

## 8. Close Database Connection

In [ ]:
conn.close()
print("Database connection closed.")

## Summary

This notebook demonstrated:

- ✅ **Database connectivity** — DuckDB opens in read-only mode without requiring a full PBI installation
- ✅ **Schema overview** — all tables (fact + dimensions) are listed with row counts
- ✅ **Phage metadata** — source distribution, genome length, GC content, lifestyle, completeness, top hosts
- ✅ **Annotation coverage** — which fraction of phages have proteins, terminators, AMR genes, etc.
- ✅ **Host files** — CSV and JSON files in `hosts/` with species and assembly metadata
- ✅ **Data-merging reports** — links to HTML reports documenting the ETL pipeline
- ✅ **Example queries** — cross-table JOINs to find phages by host, lifestyle, or annotation

## Further Resources

| Resource | URL |
|---|---|
| PBI GitHub repository | https://github.com/ThibaultSchowing/PBI |
| PBI documentation | https://thibaultschowing.github.io/PBI/ |
| DuckDB documentation | https://duckdb.org/docs/ |

## Analysis output layout (durable exports)
Generated artifacts are written to `NOTEBOOK_RESULTS_DIR`, rooted at `/results` in Docker (mounted from `./outputs`):
- `tables/` for exported DataFrames (`.parquet`)
- `figures/` for saved plots (`.png`)
This keeps source notebooks in `/workspace` clean while preserving reproducible outputs in `/results`.
Note: all currently open matplotlib figures are exported to `figures/` when running the export cell.


In [ ]:
# Export meaningful notebook artifacts to durable results storage
import pandas as pd

candidate_tables = {
    'manifest': globals().get('manifest'),
    'tables': globals().get('tables'),
    'phage_meta': globals().get('phage_meta'),
    'host_meta': globals().get('host_meta'),
    'counts_df': globals().get('counts_df'),
    'coverage_df': globals().get('coverage_df'),
    'assembly_meta': globals().get('assembly_meta'),
    'links_df': globals().get('links_df'),
    'proteins_df': globals().get('proteins_df'),
    'phage_files_df': globals().get('phage_files_df'),
    'df': globals().get('df'),
}

exported_tables = []
for name, value in candidate_tables.items():
    if isinstance(value, pd.DataFrame) and not value.empty:
        output_path = TABLES_DIR / f"{name}.parquet"
        value.to_parquet(output_path, index=False)
        exported_tables.append(output_path)

exported_figures = []
for fig_num in plt.get_fignums():
    fig_path = FIGURES_DIR / f"figure_{fig_num}.png"
    plt.figure(fig_num).savefig(fig_path, dpi=300, bbox_inches='tight')
    exported_figures.append(fig_path)

print(f"Exported {len(exported_tables)} table(s) to: {TABLES_DIR}")
for p in exported_tables:
    print(f" - {p.name}")

print(f"Exported {len(exported_figures)} figure(s) to: {FIGURES_DIR}")
for p in exported_figures:
    print(f" - {p.name}")

